# **Facial Emotion Detection**

## **Problem Definition**

**The context:** Why is this problem important to solve?<br>
**The objectives:** What is the intended goal?<br>
**The key questions:** What are the key questions that need to be answered?<br>
**The problem formulation:** What are we trying to solve using data science?

--------------
## **Context**
--------------
This problem falls under a specific branch of AI, ArtificialEmotionalIntelligence. This branch involves technologies that can read human emotions. Training a model to identify facial emotions accurately is an important step towards the development of emotionally intelligent behavior in machines with AI capabilities. Facial Emotion Recognition can have a wide variety of applications involving human and computer interaction. It can provide actionable insight in the following area:
* Market research
* Medical diagnosis
* Human safety
* Virtual or physical assistant machines

----------------
## **Objectives**
----------------

The Facial_emotion_images dataset contains over 16,000 labelled images of human facial expressions, each classified as one of 4 emotions:
* happy
* neutral
* sad
* surprise

Our objective is to use Deep Learning to predict the emotions of the test image data set by creating an accurate computer vision model.

-------------
## **Key Questions**
-------------
* Are there uniquely identifiable features of the different classes for the model to pick up on and how can we build the model to do so based on intuition
* Is the dataset sufficient and fit for purpose
* Are there any Exploratory Data Analysis tasks we can do to discover insights
* Should we use CNNs, ANNs or transfer learning techniques
* Which pre-processing or data treatments perform the best and why?
* Which model performs the best & why?
* Is there scope to improve the performance further and how?

-------------
## **Problem Formulation**
-------------
We are utilising AI & Data Science to recognize facial emotions in images by intuitively building, compiling, training & evaluating various neural network models. Some will be completely customised and trained from scratch, others will be existing computer vision models that we can use transfer learning techniques to adapt to this problem. Finally we'll select the model that is giving us the best performance and discuss scope for further improvement.

## **Mounting the Drive**

**NOTE:**   I chose to use Jupyter Notebooks with a local runtime instead of google collab because my personal computer had better performance when building models.**

## **Importing the Libraries**

In [ ]:
import zipfile
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import os

from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

import tensorflow as tf

# Importing keras Libraries
from tensorflow.keras.preprocessing.image import load_img, img_to_array, ImageDataGenerator
from tensorflow.keras.layers import Dense, Input, Dropout, GlobalAveragePooling2D, Flatten, Conv2D, BatchNormalization, Activation, MaxPooling2D, LeakyReLU
from tensorflow.keras import Model
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.optimizers import Adam, SGD, RMSprop
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import backend

In [ ]:
import warnings
# Ignore warnings
warnings.filterwarnings('ignore')

### **Let us load and unzip the data**

**Note:**
- I downloaded the dataset from the link provided on Olympus and loaded it from my local machine

In [ ]:
# Storing the path of the data file from the Google drive
path = 'C:/Users/Ryan/Documents/MIT Applied Data Science/Capstone/Facial_emotion_images.zip'

# The data is provided as a zip file so we need to extract the files from the zip file
with zipfile.ZipFile(path, 'r') as zip_ref:
    zip_ref.extractall()

## **Visualizing our Classes**

Let's look at our classes.

**Write down your observation for each class. What do you think can be a unique feature of each emotion, that separates it from the remaining classes?**

### **Happy**

In [ ]:
# Images were inspected and found to be 48*48 pixels
picture_size = 48
folder_path = "Facial_emotion_images/"

In [ ]:
# Create a function to display the top 12 images in a filepath
def top_12_images(filepath):

    # Plot size 9 width, 12 height
    plt.figure(figsize= (9,12))

    # Loop through imaes 1 to 12
    for i in range(1, 13, 1):

        # Plot with 4 rows and 3 columns
        plt.subplot(4, 3, i)

        # Load image i
        img = load_img( filepath+ '/' +
                  os.listdir(filepath)[i], target_size = (picture_size, picture_size))
        plt.imshow(img)

    plt.show()

# Plot happy images
filepath = folder_path + 'train/happy'

top_12_images(filepath)

**Observations and Insights:**
* Mouth is generally angled upwards at the corners
* Lines appear more defined between the corners of the mouth and the nose
* Eyes are generally widely openend

### **Sad**

In [ ]:
# Plot sad images
filepath = folder_path + 'train/sad'

top_12_images(filepath)

**Observations and Insights:**
* Mouth is generally angled downwards at the corners
* Lines appear less defined between the corners of the mouth and the nose
* Eyes are generally more closed and sometimes looking downward or completely closed

### **Neutral**

In [ ]:
# Plot neutral images
filepath = folder_path + 'train/neutral'

top_12_images(filepath)

**Observations and Insights:**
* Mouth is generally not angled in eithr direction at the corners
* Lines appear less defined between the corners of the mouth and the nose
* Eyes are generally opened a usual amount
* There are sometimes features of happy expressions, but not all of them simultaneously, for example, the picture in position (3,1) has more defined lines & the picture in position (3,3) has mouth corners angled upwards - but they don't have the other features to corroborate

### **Surprised**

In [ ]:
# Plot surprised images
filepath = folder_path + 'train/surprise'

top_12_images(filepath)

**Observations and Insights:**
* Mouth is generally open
* Hands sometimes on sides of face or covering mouth
* Eyes are generally opened very wide

## **Checking Distribution of Classes**

In [ ]:
# Getting count of images in each class in the training data

# Creating a function to return this value and print it in a statement
def num_class(classification):

    # Get the number of files in the classes training folder
    num = len(os.listdir(folder_path + 'train/' + classification))

    # Print this number then return it
    print('Number of images in the training data classified as ' + classification + ': ', num)
    return num

# Get the number of happy images in the training folder
num_happy = num_class('happy')

# Get the number of sad images in the training folder
num_sad = num_class('sad')

# Get the number of neutral images in the training folder
num_neutral = num_class('neutral')

# Get the number of surprised images in the training folder
num_surprise = num_class('surprise')

In [ ]:
# Plot frequency histogram of classes
plt.figure(figsize = (8, 3))

data = {'Happy': num_happy, 'Sad': num_sad, 'Neutral': num_neutral, 'Surprise' : num_surprise}

df = pd.Series(data)

plt.bar(range(len(df)), df.values, align = 'center')

plt.xticks(range(len(df)), df.index.values, size = 'small')

plt.show()

In [ ]:
# Getting count of images in each class in the validation data

# Creating a function to return this value and print it in a statement
def num_class(classification):

    # Get the number of files in the classes validation folder
    num = len(os.listdir(folder_path + 'validation/' + classification))

    # Print this number then return it
    print('Number of images in the validation data classified as ' + classification + ': ', num)
    return num

# Get the number of happy images in the validation folder
num_happy = num_class('happy')

# Get the number of sad images in the validation folder
num_sad = num_class('sad')

# Get the number of neutral images in the validation folder
num_neutral = num_class('neutral')

# Get the number of surprised images in the validation folder
num_surprise = num_class('surprise')

In [ ]:
# Plot frequency histogram of classes
plt.figure(figsize = (8, 3))

data = {'Happy': num_happy, 'Sad': num_sad, 'Neutral': num_neutral, 'Surprise' : num_surprise}

df = pd.Series(data)

plt.bar(range(len(df)), df.values, align = 'center')

plt.xticks(range(len(df)), df.index.values, size = 'small')

plt.show()

**Observations and Insights:**
* In the training data there are slightly less surprised images than the other classes, which are fairly evenly distributed. However this is likely not an issue as the surprised images still make up 21% of the total images which is a sufficient proportion when there are only 4 clsses.
* In the training data there are over 3100 images of each class which I would expect to be an ample number.
* In the validation data there are proporitionally less surprised images and more happy images, though probably still enough to acheive a reasonably performing model.
* The training:validation split is roughly 3:1 a generally sensible split.
* I think enough is known about the data at this stage to progress without further EDA, though in general it may be useful to check prperties of the images are all uniform (greyscale vs rgb & their resolution).

**Think About It:**
* Are the classes equally distributed? If not, do you think the imbalance is too high? Will it be a problem as we progress?
* Are there any Exploratory Data Analysis tasks that we can do here? Would they provide any meaningful insights?

## **Creating our Data Loaders**

In this section, we are creating data loaders that we will use as inputs to our Neural Network.

**You have two options for the color_mode. You can set it to color_mode = 'rgb' or color_mode = 'grayscale'. You will need to try out both and see for yourself which one gives better performance.**

In [ ]:
# Set the batch size
batch_size = 32
# There are only 48x48 pixels in each image
img_size = 48
# Set classes
classes = ['happy', 'neutral', 'sad', 'surprise']

# Create the Data Loader for training set
# flipping horizontally should be inconsequential to classification
datagen_train = ImageDataGenerator(
    horizontal_flip = True,
    brightness_range=(0.,2.),
    rescale=1./255,
    shear_range=0.3)

# Create the train set
# color_mode = 'grayscale' chosen since imges are grayscale anyway so we shouldn't lose accuracy, however performance should be better
train_set = datagen_train.flow_from_directory(
    folder_path + "train",
    target_size = (img_size, img_size),
    color_mode = 'grayscale',
    batch_size = batch_size,
    class_mode = 'categorical',
    classes = classes,
    shuffle = True
)

# Create the Data Loader for validation/test sets
datagen_validation = ImageDataGenerator(rescale=1./255)

# Create the validation set
validation_set = datagen_validation.flow_from_directory(
    folder_path + "validation",
    target_size = (img_size, img_size),
    color_mode = 'grayscale',
    batch_size = batch_size,
    class_mode = 'categorical',
    classes = classes,
    shuffle = True
)

# Create the test set
test_set = datagen_validation.flow_from_directory(
    folder_path + "test",
    target_size = (img_size, img_size),
    color_mode = 'grayscale',
    batch_size = batch_size,
    class_mode = 'categorical',
    classes = classes,
    shuffle = False
)

## **Model Building**

**Think About It:**
* Are Convolutional Neural Networks the right approach? Should we have gone with Artificial Neural Networks instead?
* What are the advantages of CNNs over ANNs and are they applicable here?

**Answers**
* CNNs are generally a superior approach for computer vision tasks when compared to ANNs
* The main advantage of CNNs is that they automatically detect the important features without any human supervision.
* ANNs on the other hand, treat the input data as a vector (a sequence of numbers). They do not take into account the spatial relationships between pixels in the image and therefore lose some ability to learn features that can only be defined in terms of 2D space.

### **Creating the Base Neural Network**

In [ ]:
# Define the model
def cnn_model_1():

    # Initializing a Sequential Model
    model = Sequential()

    model.add(tf.keras.layers.Conv2D(filters=64, kernel_size=(2,2), padding='same', activation='relu', input_shape = (48, 48, 1)))

    model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))

    model.add(tf.keras.layers.Dropout(.2))

    model.add(tf.keras.layers.Conv2D(filters=32, kernel_size=(2,2), padding='same', activation='relu'))

    model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))

    model.add(tf.keras.layers.Dropout(.2))

    model.add(tf.keras.layers.Conv2D(filters=32, kernel_size=(2,2), padding='same', activation='relu'))

    model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))

    model.add(tf.keras.layers.Dropout(.2))

    model.add(tf.keras.layers.Flatten())

    model.add(tf.keras.layers.Dense(512, activation='relu'))

    model.add(tf.keras.layers.Dropout(.4))

    model.add(tf.keras.layers.Dense(units=4, activation='softmax'))

    return model

cnn_model_1 = cnn_model_1()

cnn_model_1.summary()

### **Compiling and Training the Model**

In [ ]:
# Compile the model
cnn_model_1.compile(optimizer = 'adam', loss = 'categorical_crossentropy', metrics = ['accuracy'])

# Train the model
epochs = 20

early_stopping = EarlyStopping(
    monitor = 'val_loss',
    min_delta = 0,
    patience = 3,
    verbose = 1,
    restore_best_weights = True
)

checkpoint = ModelCheckpoint("./cnn_model_1.h5", monitor='val_accuracy', verbose=1, save_best_only=True, mode='max')

reduce_learningrate = ReduceLROnPlateau(
    monitor = 'val_loss',
    factor = 0.2,
    patience = 3,
    verbose = 1,
    min_delta = 0.0001
)

callbacks_list = [early_stopping, checkpoint, reduce_learningrate]

cnn_model_1_fit = cnn_model_1.fit(
    x               = train_set,
    batch_size      = batch_size,
    verbose         = 1,
    epochs          = epochs,
    callbacks       = callbacks_list,
    validation_data = validation_set
)

### **Evaluating the Model on the Test Set**

In [ ]:
def plot_confusion_matrix(model, test_set):

    labels = ['happy', 'neutral', 'sad', 'surprise']

    # Predict on the test data & convert each entry to integers
    test_pred = np.argmax(model.predict(test_set), axis = -1)

    # Convert to string labels
    conditions = [test_pred == 0, test_pred == 1, test_pred == 2, test_pred == 3]
    test_pred_labels = np.select(conditions, labels)

    # Converting each entry to integers
    test_true = test_set.labels

    # Convert to string labels
    conditions = [test_true == 0, test_true == 1, test_true == 2, test_true == 3]
    test_true_labels = np.select(conditions, labels)

    # Printing the classification report
    print(classification_report(y_true=test_true_labels, y_pred=test_pred_labels))

    # Creating confusion matrix using actual labels (test_true) and predicted labels (test_pred)
    cm = confusion_matrix(test_true, test_pred)

    # set plot size
    plt.figure(figsize = (8, 5))

    # create heatmap using confusion matrix
    sns.heatmap(cm, annot = True, xticklabels = labels, yticklabels = labels)

    # set labels
    plt.ylabel('Actual')
    plt.xlabel('Predicted')

    plt.show()

plot_confusion_matrix(cnn_model_1, test_set)

**Observations and Insights:**
* 0.68 f-score is not bad for an initial model and this may not have plateaued at 20 epochs, so perhaps I will increase # of epochs in a future model.
* This model struggles to separate sad and neutral emotions
* Suprirse & Happy emoptions are categorised fairly accurately, though accuracy could be improved.

### **Creating the second Convolutional Neural Network**

- Try out a slightly larger architecture

In [ ]:
# Define the model
def cnn_model_2():

    # Initializing a Sequential Model
    model = Sequential()

    model.add(tf.keras.layers.Conv2D(filters=256, kernel_size=(2,2), padding='same', activation='relu', input_shape = (48, 48, 1)))

    model.add(tf.keras.layers.BatchNormalization())

    model.add(tf.keras.layers.LeakyReLU(alpha=0.1))

    model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))

    model.add(tf.keras.layers.Conv2D(filters=128, kernel_size=(2,2), padding='same', activation='relu'))

    model.add(tf.keras.layers.BatchNormalization())

    model.add(tf.keras.layers.LeakyReLU(alpha=0.1))

    model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))

    model.add(tf.keras.layers.Conv2D(filters=64, kernel_size=(2,2), padding='same', activation='relu'))

    model.add(tf.keras.layers.BatchNormalization())

    model.add(tf.keras.layers.LeakyReLU(alpha=0.1))

    model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))

    model.add(tf.keras.layers.Conv2D(filters=32, kernel_size=(2,2), padding='same', activation='relu'))

    model.add(tf.keras.layers.BatchNormalization())

    model.add(tf.keras.layers.LeakyReLU(alpha=0.1))

    model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))

    model.add(tf.keras.layers.Flatten())

    model.add(tf.keras.layers.Dense(512, activation='relu'))

    model.add(tf.keras.layers.Dense(128, activation='relu'))

    model.add(tf.keras.layers.Dropout(.4))

    model.add(tf.keras.layers.Dense(units=4, activation='softmax'))

    return model

cnn_model_2 = cnn_model_2()

cnn_model_2.summary()

### **Compiling and Training the Model**

In [ ]:
# Compile the model
cnn_model_2.compile(optimizer = 'adam', loss = 'categorical_crossentropy', metrics = ['accuracy'])

# Train the model
epochs = 20

early_stopping = EarlyStopping(
    monitor = 'val_loss',
    min_delta = 0,
    patience = 3,
    verbose = 1,
    restore_best_weights = True
)

checkpoint = ModelCheckpoint("./cnn_model_2.h5", monitor='val_accuracy', verbose=1, save_best_only=True, mode='max')

reduce_learningrate = ReduceLROnPlateau(
    monitor = 'val_loss',
    factor = 0.2,
    patience = 3,
    verbose = 1,
    min_delta = 0.0001
)

callbacks_list = [early_stopping, checkpoint, reduce_learningrate]

cnn_model_2_fit = cnn_model_2.fit(
    x               = train_set,
    batch_size      = batch_size,
    verbose         = 1,
    epochs          = epochs,
    callbacks       = callbacks_list,
    validation_data = validation_set
)

### **Evaluating the Model on the Test Set**

In [ ]:
# Plot confusion matrix
plot_confusion_matrix(cnn_model_2, test_set)

**Observations and Insights:**
* 0.70 accuracy and 0.71 f-score is a slight improvement on the initial model and this may not have plateaued at 20 epochs, so perhaps I will increase # of epochs in a future model.
* This model again struggles to separate sad and neutral emotions.
* Suprirse & Happy emoptions are categorised very accurately

## **Think About It:**

* Did the models have a satisfactory performance? If not, then what are the possible reasons?
* Which Color mode showed better overall performance? What are the possible reasons? Do you think having 'rgb' color mode is needed because the images are already black and white?

**Answers**
* The second model is the better performing one. It provided performance that may be useful for some applications where false negatives are not important, especially for recognising happy and surprised images. However, it doesn't perform sufficiently well for neutral and sad emotions.
* Greyscale performs better, because the images are already grescale so we don't lose any information or gain any superfluous information or more complex structure.

## **Transfer Learning Architectures**

In this section, we will create several Transfer Learning architectures. For the pre-trained models, we will select three popular architectures namely, VGG16, ResNet v2, and Efficient Net. The difference between these architectures and the previous architectures is that these will require 3 input channels while the earlier ones worked on 'grayscale' images. Therefore, we need to create new DataLoaders.

### **Creating our Data Loaders for Transfer Learning Architectures**

In this section, we are creating data loaders that we will use as inputs to our Neural Network. We will have to go with color_mode = 'rgb' as this is the required format for the transfer learning architectures.

In [ ]:
batch_size  = 32
img_size = 48

classes = ['happy', 'neutral', 'sad', 'surprise']

datagen_train = ImageDataGenerator(horizontal_flip = True,
                                    brightness_range = (0., 2.),
                                    rescale = 1./255,
                                    shear_range = 0.3)

train_set = datagen_train.flow_from_directory(
    folder_path + "train",
    target_size = (img_size, img_size),
    color_mode = 'rgb',
    batch_size = batch_size,
    class_mode = 'categorical',
    classes = classes,
    shuffle = True
)

# Create the Data Loader for validation/test sets
datagen_validation = ImageDataGenerator(rescale=1./255)

# Create the validation set
validation_set = datagen_validation.flow_from_directory(
    folder_path + "validation",
    target_size = (img_size, img_size),
    color_mode = 'rgb',
    batch_size = batch_size,
    class_mode = 'categorical',
    classes = classes,
    shuffle = True
)

test_set = datagen_validation.flow_from_directory(
    folder_path + "test",
    target_size = (img_size, img_size),
    color_mode = 'rgb',
    batch_size = batch_size,
    class_mode = 'categorical',
    classes = classes,
    shuffle = False
)

## **VGG16 Model**

### **Importing the VGG16 Architecture**

In [ ]:
from tensorflow.keras.applications.vgg16 import VGG16

vgg = VGG16(include_top = False, weights = 'imagenet', input_shape = (48, 48, 3))
vgg.summary()

### **Model Building**

- Import VGG16 upto the layer of your choice and add Fully Connected layers on top of it.

In [ ]:
transfer_layer = vgg.get_layer('block5_pool')
vgg.trainable = False

# Flattenning the output from the 3rd block of the VGG16 model
x = Flatten()(transfer_layer.output)

# Adding a Dense layer with 256 neurons
x = Dense(256, activation = 'relu')(x)

# Add a Dense Layer with 128 neurons
x = Dense(128, activation = 'relu')(x)

# Add a DropOut layer with Drop out ratio of 0.3
x = Dropout(0.3)(x)

# Add a Dense Layer with 64 neurons
x = Dense(64, activation = 'relu')(x)

# Add a Batch Normalization layer
x = BatchNormalization()(x)

# Adding the final dense layer with 4 neurons and use 'softmax' activation
pred = Dense(4, activation='softmax')(x)

vggmodel = Model(vgg.input, pred) # Initializing the model

### **Compiling and Training the VGG16 Model**

In [ ]:
checkpoint = ModelCheckpoint("./vggmodel.h5", monitor = 'val_accuracy', verbose = 1, save_best_only = True, mode = 'max')

early_stopping = EarlyStopping(
    monitor = 'val_loss',
    min_delta = 0,
    patience = 3,
    verbose = 1,
    restore_best_weights = True
)

reduce_learningrate = ReduceLROnPlateau(
    monitor = 'val_loss',
    factor = 0.2,
    patience = 3,
    verbose = 1,
    min_delta = 0.0001
)

callbacks_list = [early_stopping, checkpoint, reduce_learningrate]

epochs = 20

#compile the model
vggmodel.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
#train your model on data
vggmodel_fit = vggmodel.fit(
    x               = train_set,
    batch_size      = batch_size,
    verbose         = 1,
    epochs          = epochs,
    callbacks       = callbacks_list,
    validation_data = validation_set
)

### **Evaluating the VGG16 model**

In [ ]:
# Plotting the accuracies

# store history
vggmodel_hist = vggmodel_fit.history

# get a list of 20 epochs
list_ep = [i for i in range(1, 21)]

# set plot size
plt.figure(figsize = (9, 9))

# plot accuracy against epochs with labels
plt.plot(list_ep, vggmodel_hist['accuracy'], ls = '--', label = 'accuracy')
plt.plot(list_ep, vggmodel_hist['val_accuracy'], ls = '--', label = 'val_accuracy')

# set axis labels
plt.ylabel('Accuracy')
plt.xlabel('Epochs')

plt.legend()

plt.show()

In [ ]:
# Plot confusion matrix
plot_confusion_matrix(vggmodel, test_set)

**Think About It:**

- What do you infer from the general trend in the training performance?
- Is the training accuracy consistently improving?
- Is the validation accuracy also improving similarly?

**Observations and Insights:**
* The training performance is improving over increasing epochs, though I would anticipate that it wouldn't get much higher accuracy on the validation set which seems to have plateaued or be close to it.
* Training accuracy was consistently improving up to 20 epochs
* Validation accuracy was improving similarly but with more variance

**Note: You can even go back and build your own architecture on top of the VGG16 Transfer layer and see if you can improve the performance**

## **ResNet V2 Model**

In [ ]:
import tensorflow.keras.applications as ap

Resnet = ap.ResNet101(include_top = False, weights = "imagenet", input_shape=(48,48,3))
Resnet.summary()

### **Model Building**

- Import Resnet v2 upto the layer of your choice and add Fully Connected layers on top of it.

In [ ]:
transfer_layer_Resnet = Resnet.get_layer('conv5_block3_add')
Resnet.trainable=False

# 1st fully connected block
x = Dense(256, activation='relu')(transfer_layer_Resnet.output)

x = BatchNormalization()(x)

x = LeakyReLU(alpha=0.1)(x)

x = Dropout(.2)(x)

# 2nd fully connected block
x = Dense(512, activation='relu')(x)

x = BatchNormalization()(x)

x = LeakyReLU(alpha=0.1)(x)

x = Dropout(.2)(x)

# Flatten
x = Flatten()(x)

# Add a Dense layer with 256 neurons
x = Dense(256, activation = 'relu')(x)

# Add a Dense Layer with 128 neurons
x = Dense(128, activation = 'relu')(x)

# Add a DropOut layer with Drop out ratio of 0.3
x = Dropout(0.3)(x)

# Add a Dense Layer with 64 neurons
x = Dense(64, activation = 'relu')(x)

# Add a Batch Normalization layer
x = BatchNormalization()(x)

# Add the final dense layer with 4 neurons and use a 'softmax' activation
pred = Dense(4, activation = 'softmax')(x)

resnetmodel = Model(Resnet.input, pred) # Initializing the model

### **Compiling and Training the Model**

In [ ]:
# Compile the model
resnetmodel.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
epochs = 20
checkpoint = ModelCheckpoint("./Resnetmodel.h5", monitor = 'val_accuracy', verbose = 1, save_best_only = True, mode = 'max')

early_stopping = EarlyStopping(
    monitor = 'val_loss',
    min_delta = 0,
    patience = 3,
    verbose = 1,
    restore_best_weights = True
)

reduce_learningrate = ReduceLROnPlateau(
    monitor = 'val_loss',
    factor = 0.2,
    patience = 3,
    verbose = 1,
    min_delta = 0.0001
)

callbacks_list = [early_stopping, checkpoint, reduce_learningrate]

resnetmodel_fit = resnetmodel.fit(
    x               = train_set,
    batch_size      = batch_size,
    verbose         = 1,
    epochs          = epochs,
    callbacks       = callbacks_list,
    validation_data = validation_set
)

### **Evaluating the ResNet Model**

In [ ]:
# Plotting the accuracies

# store history
resnetmodel_hist = resnetmodel_fit.history

# get a list of 20 epochs
list_ep = [i for i in range(1, 11)]

# set plot size
plt.figure(figsize = (9, 9))

# plot accuracy against epochs with labels
plt.plot(list_ep, resnetmodel_hist['accuracy'], ls = '--', label = 'accuracy')
plt.plot(list_ep, resnetmodel_hist['val_accuracy'], ls = '--', label = 'val_accuracy')

# set axis labels
plt.ylabel('Accuracy')
plt.xlabel('Epochs')

plt.legend()

plt.show()

In [ ]:
# Plot Confusuion Matrix
plot_confusion_matrix(resnetmodel,test_set)

**Observations and Insights:**
* The training performance is improving over increasing epochs, though it is not very accurate at all & the validation performance is sporadic and has no trend
* Training accuracy was consistently improving up to 10 epochs
* Validation accuracy was terrible.

**Note: You can even go back and build your own architecture on top of the ResNet Transfer layer and see if you can improve the performance.**

## **EfficientNet Model**

In [ ]:
# Create base model
EfficientNet = ap.EfficientNetV2B2(include_top=False, weights="imagenet", input_shape= (48, 48, 3))
EfficientNet.summary()

### **Model Building**

- Import EfficientNet upto the layer of your choice and add Fully Connected layers on top of it.

In [ ]:
transfer_layer_EfficientNet = EfficientNet.get_layer('block6j_expand_activation')
EfficientNet.trainable = False

# 1st fully connected block
x = Dense(256, activation='relu')(transfer_layer_EfficientNet.output)

x = BatchNormalization()(x)

x = LeakyReLU(alpha=0.1)(x)

x = Dropout(.2)(x)

# 2nd fully connected block
x = Dense(512, activation='relu')(x)

x = BatchNormalization()(x)

x = LeakyReLU(alpha=0.1)(x)

x = Dropout(.2)(x)

# Add your Flatten layer.
x = Flatten()(x)

# Add a Dense layer with 256 neurons
x = Dense(256, activation = 'relu')(x)

# Add a Dense Layer with 128 neurons
x = Dense(128, activation = 'relu')(x)

# Add a DropOut layer with Drop out ratio of 0.3
x = Dropout(0.3)(x)

# Add a Dense Layer with 64 neurons
x = Dense(64, activation = 'relu')(x)

# Add a Batch Normalization layer
x = BatchNormalization()(x)

# Add your final Dense layer with 4 neurons and softmax activation function.
pred = Dense(4, activation = 'softmax')(x)

# Intitialise model
Efficientnetmodel = Model(EfficientNet.input, pred)

### **Compiling and Training the Model**

In [ ]:
# Compile the model
Efficientnetmodel.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
epochs = 20

checkpoint = ModelCheckpoint("./Efficientnetmodel.h5", monitor = 'val_accuracy', verbose = 1, save_best_only = True, mode = 'max')

early_stopping = EarlyStopping(
    monitor = 'val_loss',
    min_delta = 0,
    patience = 3,
    verbose = 1,
    restore_best_weights = True
)

reduce_learningrate = ReduceLROnPlateau(
    monitor = 'val_loss',
    factor = 0.2,
    patience = 3,
    verbose = 1,
    min_delta = 0.0001
)

callbacks_list = [early_stopping, checkpoint, reduce_learningrate]

Efficientnetmodel_fit = Efficientnetmodel.fit(
    x               = train_set,
    batch_size      = batch_size,
    verbose         = 1,
    epochs          = epochs,
    callbacks       = callbacks_list,
    validation_data = validation_set
)

### **Evaluating the EfficientnetNet Model**

In [ ]:
# Plotting the accuracies

# store history
Efficientnetmodel_hist = Efficientnetmodel_fit.history

# get a list of 20 epochs
list_ep = [i for i in range(1, 10)]

# set plot size
plt.figure(figsize = (9, 9))

# plot accuracy against epochs with labels
plt.plot(list_ep, Efficientnetmodel_hist['accuracy'], ls = '--', label = 'accuracy')
plt.plot(list_ep, Efficientnetmodel_hist['val_accuracy'], ls = '--', label = 'val_accuracy')

# set axis labels
plt.ylabel('Accuracy')
plt.xlabel('Epochs')

plt.legend()

plt.show()

In [ ]:
# Plot confusion matrix
plot_confusion_matrix(Efficientnetmodel,test_set)

**Observations and Insights:**
* This model categorises everything as happy. It is useless

**Think About It:**

* What is your overall performance of these Transfer Learning Architectures? Can we draw a comparison of these models' performances. Are we satisfied with the accuracies that we have received?
* Do you think our issue lies with 'rgb' color_mode?

**Answers**
* One of the models based on VGG16 was not completely useless, though it still has dissatisfactory performance when compared to the previous CNN models.
* The other models were useless.
* The issue could well be with the rgb color_mode being utilised and the mdels being pre-trained on the when the images are greyscale
* It's worht noting that we did not perform fine-tuning, which consists of unfreezing the entire model (or part of it), and re-training it on the new data with a very low learning rate. This can potentially achieve meaningful improvements, by incrementally adapting the pretrained features to the new data. Howewever it didn't seem worth it when performance was already so inferior to the other CNN models we built.

Now that we have tried multiple pre-trained models, let's build a complex CNN architecture and see if we can get better performance.

## **Building a Complex Neural Network Architecture**

In this section, we will build a more complex Convolutional Neural Network Model that has close to as many parameters as we had in our Transfer Learning Models. However, we will have only 1 input channel for our input images.

## **Creating our Data Loaders**

In this section, we are creating data loaders which we will use as inputs to the more Complicated Convolutional Neural Network. We will go ahead with color_mode = 'grayscale'.

In [ ]:
batch_size  = 32
img_size = 48
classes = ['happy', 'neutral', 'sad', 'surprise']

datagen_train = ImageDataGenerator(
    horizontal_flip = True,
    brightness_range = (0., 2.),
    rescale = 1./255,
    shear_range = 0.3
)

train_set = datagen_train.flow_from_directory(
    folder_path + "train",
    target_size = (img_size, img_size),
    color_mode = 'grayscale',
    batch_size = batch_size,
    class_mode = 'categorical',
    classes = classes,
    shuffle = True
)

datagen_validation = ImageDataGenerator(rescale = 1./255)

validation_set = datagen_validation.flow_from_directory(
    folder_path + "validation",
    target_size = (img_size, img_size),
    color_mode = 'grayscale',
    batch_size = batch_size,
    class_mode = 'categorical',
    classes = classes,
    shuffle = True
)

test_set = datagen_validation.flow_from_directory(
    folder_path + "test",
    target_size = (img_size, img_size),
    color_mode = 'grayscale',
    batch_size = batch_size,
    class_mode = 'categorical',
    classes = classes,
    shuffle = False
)

### **Model Building**

- Try building a layer with 5 Convolutional Blocks and see if performance increases.

In [ ]:
# Define the model
def cnn_model_3():

    # Initializing a Sequential Model
    model = Sequential()

    # Add 1st CNN Block
    model.add(tf.keras.layers.Conv2D(filters=64, kernel_size=(2,2), padding='same', activation='relu', input_shape = (48, 48, 1)))

    model.add(tf.keras.layers.BatchNormalization())

    model.add(tf.keras.layers.LeakyReLU(alpha=0.1))

    model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))

    model.add(tf.keras.layers.Dropout(.2))

    # Add 2nd CNN Block
    model.add(tf.keras.layers.Conv2D(filters=128, kernel_size=(2,2), padding='same', activation='relu'))

    model.add(tf.keras.layers.BatchNormalization())

    model.add(tf.keras.layers.LeakyReLU(alpha=0.1))

    model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))

    model.add(tf.keras.layers.Dropout(.2))

    # Add 3rd CNN Block
    model.add(tf.keras.layers.Conv2D(filters=512, kernel_size=(2,2), padding='same', activation='relu'))

    model.add(tf.keras.layers.BatchNormalization())

    model.add(tf.keras.layers.LeakyReLU(alpha=0.1))

    model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))

    model.add(tf.keras.layers.Dropout(.2))

    # Add 4th CNN Block
    model.add(tf.keras.layers.Conv2D(filters=512, kernel_size=(2,2), padding='same', activation='relu'))

    model.add(tf.keras.layers.BatchNormalization())

    model.add(tf.keras.layers.LeakyReLU(alpha=0.1))

    model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))

    model.add(tf.keras.layers.Dropout(.2))

    # Add 5th CNN Block
    model.add(tf.keras.layers.Conv2D(filters=512, kernel_size=(2,2), padding='same', activation='relu'))

    model.add(tf.keras.layers.BatchNormalization())

    model.add(tf.keras.layers.LeakyReLU(alpha=0.1))

    model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))

    model.add(tf.keras.layers.Dropout(.2))

    # Flatten
    model.add(tf.keras.layers.Flatten())

    # 1st fully connected block
    model.add(tf.keras.layers.Dense(256, activation='relu'))

    model.add(tf.keras.layers.BatchNormalization())

    model.add(tf.keras.layers.LeakyReLU(alpha=0.1))

    model.add(tf.keras.layers.Dropout(.2))

    # 2nd fully connected block
    model.add(tf.keras.layers.Dense(512, activation='relu'))

    model.add(tf.keras.layers.BatchNormalization())

    model.add(tf.keras.layers.LeakyReLU(alpha=0.1))

    model.add(tf.keras.layers.Dropout(.2))

    # 3rd fully connected block
    model.add(tf.keras.layers.Dense(512, activation='relu'))

    model.add(tf.keras.layers.BatchNormalization())

    model.add(tf.keras.layers.LeakyReLU(alpha=0.1))

    model.add(tf.keras.layers.Dropout(.2))

    # Final classification
    model.add(tf.keras.layers.Dense(units=4, activation='softmax'))

    return model

cnn_model_3 = cnn_model_3()

cnn_model_3.summary()

### **Compiling and Training the Model**

In [ ]:
# Compile the model
cnn_model_3.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
epochs = 35

steps_per_epoch = train_set.n//train_set.batch_size
validation_steps = validation_set.n//validation_set.batch_size

checkpoint = ModelCheckpoint("cnn_model_3.h5", monitor = 'val_accuracy',
                            save_weights_only = True, model = 'max', verbose = 1)

reduce_lr = ReduceLROnPlateau(monitor = 'val_loss', factor = 0.1, patience = 2, min_lr = 0.0001 , model = 'auto')

callbacks = [checkpoint, reduce_lr]

cnn_model_3_fit = cnn_model_3.fit(
    x                = train_set,
    batch_size       = batch_size,
    verbose          = 1,
    epochs           = epochs,
    callbacks        = callbacks,
    validation_data  = validation_set,
    steps_per_epoch  = steps_per_epoch,
    validation_steps = validation_steps
)

### **Evaluating the Model on Test Set**

In [ ]:
# Plotting the accuracies

# store history
cnn_model_3_hist = cnn_model_3_fit.history

# get a list of 20 epochs
list_ep = [i for i in range(1, 36)]

# set plot size
plt.figure(figsize = (9, 9))

# plot accuracy against epochs with labels
plt.plot(list_ep, cnn_model_3_hist['accuracy'], ls = '--', label = 'accuracy')
plt.plot(list_ep, cnn_model_3_hist['val_accuracy'], ls = '--', label = 'val_accuracy')

# set axis labels
plt.ylabel('Accuracy')
plt.xlabel('Epochs')

plt.legend()

plt.show()

In [ ]:
# Plot confusion matrix
plot_confusion_matrix(cnn_model_3,test_set)

**Observations and Insights:**
* 0.75 accuracy and 0.75 f-score is a decent improvement on the 2nd CNN model
* This may have plateaued at 35 epochs, if not it looks close to it.
* This model again struggles to accurately categorise sad and neutral emotions, though this may be a problem with labelling in the dataset, upon further inspection.
* Additionally, there can be a lot of overlap between the features of sad and neutral emoitional expression which makes this subset of the classification problem more difficult and maybe there is less scope to acheive as high of a score here.
* Suprirse & Happy emoptions are categorised very accurately and this model would certainly be useful for applications reuiring that.

### **Plotting the Confusion Matrix for the chosen final model**

In [ ]:
# Plot confusion matrix
plot_confusion_matrix(cnn_model_3,test_set)

**Observations and Insights:**
* cnn_model_3 is the best model, however, it could be substantially improved in terms of sad and neutral emotion detection

## **Conclusion:**
cnn_model_3 is the best model we tested, grayscale is the superior color_mode, further scope for improvement is expected.

### **Insights**

### **Refined insights**:
- What are the most meaningful insights from the data relevant to the problem?

CNNs are generally a superior approach for computer vision tasks when compared to ANNs because they automatically detect the important features without any human supervision, especially features involving 2D spatial relationships between pixels in the image.
Classes are not entireley evenly distributed, but evenly enough to acheive good results, there are more easily identifiable features in the happy and surprised clasees than in the neutral and sad classes.
Grayscale is the better performing colorscale on this dataset likely because the images are already grescale so we don't lose any information or gain any superfluous information or more complex structure thus creaing a more efficient and specific model.
Transfer learning models had poorer performance than other CNN models, perhaps how they had been built and trained was not representative of this problem and dataset, one aspect of that being the rgb colorscale they used. They did not help improve the performance.
Hence building more intuitively designed CNNs was preferred. The intuition being that larger numbers of nodes at layers (both Conv2D & Dense layers), would allow us to capture larger spatial features that relied on a greater areas of the image (e.g. eyebrows, mouth, hands and other large features).


### **Comparison of various techniques and their relative performance**:
- How do different techniques perform? Which one is performing relatively better? Is there scope to improve the performance further?

#### **Ranking models:**
1. cnn_model_3 - 0.75 accuracy & 0.75 f1-score - particularly higher precision for happy and surprise classes (0.82 & 0.93 respectively)
2. cnn_model_2 - 0.70 accuracy & 0.71 f1-score
3. cnn_model_1 - 0.68 accuracy & 0.68 f1-score
4. vggmodel - 0.52 accuracy & 0.52 f1-score
5. resnetmodel - completely useless - very low scores
6. Efficientnetmodel - completely useless - predicts everything to be happy

Again, the Transfer learning models had poorer performance than other CNN models, perhaps how they had been built and trained was not representative of this problem and dataset, one aspect of that being the rgb colorscale they used.
The classification report & confusion matrix for the final model cnn_model_3 can be found just above for reference.

#### **Scope for performance improvement:**
* We can have a colour image dataset with higher resolution and using the rgb colorscale, however ths will come at a cost of performance and we might need to use additional techniques to manage the size of the high resolution images.
* We can use k fold cross validation on a combination of the training and test dataset.
* We can by review the labelling of the datasets, especially the sad and neutral labelled images as they had the most miscategorisation and upon quickly scanning through I could see several labels that I disagree with. Relabelling more accurately couldd help performance
* We can further manipulate the hyperparameters and model structure of the complex CNN model.

### **Proposal for the final solution design**:
- What model do you propose to be adopted? Why is this the best solution to adopt?

cnn_model_3 is the best model we tested and is the chosenfinal model for it's superior scores across the board: accuracy, precision & recall are higher than all other models for all classes.
The intuition behind the better performance is that the larger numbers of nodes at various layers of the neural network (both Conv2D & Dense layers), allow us to capture larger spatial features that relied on a greater areas of the image (e.g. eyebrows, mouth, hands and other large features).